# Fit action planning model

September 2025

In [1]:
import jax
import jax.numpy as jnp
import optax

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import model

sns.set_theme()

## Model fitting

<!-- Base model (H0): $p(a|c) \propto \exp(\alpha \cdot (-w_r \cdot c_{\text{risk}}(a) - w_e \cdot c_{\text{effort}}(a)))$

Relationship model (H1): $p(a|c) \propto \exp(\alpha \cdot (-w_d \cdot c_{\text{discomfort}}(a|c) - w_r \cdot c_{\text{risk}}(a) - w_e \cdot c_{\text{effort}}(a)))$ -->

### Load action planning data

In [2]:
def load_data():
    """Load and prepare the planning data"""
    print("Loading planning data...")
    data = pd.read_csv("../data/planning-1/main_trials_tidy.csv")
    data.insert(
        data.columns.get_loc("scenario_label") + 2,
        "scenario_idx",
        data["scenario_label"].apply(lambda x: model.scenario_labels.index(x))
    )
    
    data_scenario_idx = jnp.array(data["scenario_idx"].values)
    data_action = jnp.array(data["action"].values)
    data_closeness = jnp.array(data["closeness"].values)
    data_p_action = jnp.array(data["p_action"].values)
    
    print(f"Loaded {len(data)} data points")
    return data, data_scenario_idx, data_action, data_closeness, data_p_action

In [3]:
data, data_scenario_idx, data_action, data_closeness, data_p_action = load_data()

Loading planning data...
Loaded 5696 data points


In [4]:
@jax.jit
def compute_NLL(preds, responses):
    epsilon = 1e-8
    preds_safe = jnp.clip(preds, epsilon, 1.0)
    responses_safe = jnp.clip(responses, epsilon, 1.0)
    nll = -jnp.sum(responses_safe * jnp.log(preds_safe))
    return nll

In [5]:
def fit_params(
    model_type,
    initial_params,
    data_scenario_idx,
    data_action,
    data_closeness,
    lr=0.001,
    tol=1e-6,
    max_steps=5000,
):
    """Fit model parameters using gradient descent"""
    predict_fn = model.get_vmap_predictor(model_type)

    def loss_fn(params):
        preds = predict_fn(data_scenario_idx, data_action, data_closeness, *params)
        return compute_NLL(preds, data_p_action)

    params = initial_params
    grad_fn = jax.value_and_grad(loss_fn)
    opt = optax.adam(learning_rate=lr)
    opt_state = opt.init(params)

    prev_nll = None 
    for step in range(max_steps):
        nll, grad = grad_fn(params)
        updates, opt_state = opt.update(grad, opt_state)
        params = optax.apply_updates(params, updates)

        params = params.at[:].set(jnp.clip(params[:], 0, jnp.inf)) # ensure params are positive

        if step % 1000 == 0:
            print(f"Step {step}, NLL: {nll}, params: {params}")

        if prev_nll is not None and nll > prev_nll:
            print(f"NLL increased at step {step}, stopping")
            break

        prev_nll = nll

    best_nll = loss_fn(params)

    return params, best_nll

First, fit params for vanilla model

In [6]:
params_and_nlls = {}

In [7]:
vanilla_params, vanilla_nll = fit_params(
    model_type="vanilla",
    initial_params=jnp.array([1.0, 1.0, 1.0]), # last argument is w_c, which is not used for vanilla model
    data_scenario_idx=data_scenario_idx,
    data_action=data_action,
    data_closeness=data_closeness,
)
print(f"Best NLL for vanilla model: {vanilla_nll}")
print(f"Best params for vanilla model: alpha: {vanilla_params[0]}, w_r: {vanilla_params[1]}")

params_and_nlls["vanilla"] = (vanilla_params, vanilla_nll)


Step 0, NLL: 2586.11767578125, params: [0.999 0.999 1.   ]
Step 1000, NLL: 1920.675537109375, params: [0.49372387 0.49372387 1.        ]
NLL increased at step 1725, stopping
Best NLL for vanilla model: 1916.74267578125
Best params for vanilla model: alpha: 0.4430391490459442, w_r: 0.4430391490459442


Freeze vanilla params, fit relationship model optimizing only weight on closeness $w_c$

In [8]:
# Freeze best-fitting vanilla params (alpha, w_r) and optimize only w_c for relationship model
fixed_alpha = float(vanilla_params[0])
fixed_w_r = float(vanilla_params[1])

@jax.jit
def relationship_nll_with_wc(w_c_scalar):
    preds = model.predict_relationship_vmap(
        data_scenario_idx,
        data_action,
        data_closeness,
        fixed_alpha,
        fixed_w_r,
        w_c_scalar,
    )
    return compute_NLL(preds, data_p_action)

# Optimize scalar w_c using Adam
w_c = jnp.array(1.0)
opt = optax.adam(learning_rate=0.001)
opt_state = opt.init(w_c)

prev_nll = None
grad_fn = jax.value_and_grad(relationship_nll_with_wc)

for step in range(10000):
    nll, grad = grad_fn(w_c)
    updates, opt_state = opt.update(grad, opt_state)
    w_c = optax.apply_updates(w_c, updates)

    w_c = jnp.clip(w_c, 0.0, jnp.inf)

    if step % 1000 == 0:
        print(f"Step {step}, NLL: {float(nll):.6f}, w_c: {float(w_c):.6f}")

    if prev_nll is not None and nll > prev_nll:
        print(f"NLL increased at step {step}, stopping")
        break

    prev_nll = nll

best_w_c_relationship = w_c
best_nll_relationship = relationship_nll_with_wc(w_c)

print(f"Best NLL for relationship (alpha,w_r frozen): {float(best_nll_relationship):.6f}")
print(f"Frozen alpha: {fixed_alpha:.6f}, Frozen w_r: {fixed_w_r:.6f}, Best w_c: {float(best_w_c_relationship):.6f}")

params_and_nlls["relationship"] = (jnp.array([fixed_alpha, fixed_w_r, best_w_c_relationship]), best_nll_relationship)


Step 0, NLL: 2198.250732, w_c: 1.001000
Step 1000, NLL: 1980.657715, w_c: 1.670662
Step 2000, NLL: 1962.760986, w_c: 1.952038
Step 3000, NLL: 1958.843140, w_c: 2.130109
Step 4000, NLL: 1957.544922, w_c: 2.263983
Step 5000, NLL: 1957.007202, w_c: 2.375278
NLL increased at step 5328, stopping
Best NLL for relationship (alpha,w_r frozen): 1956.903198
Frozen alpha: 0.443039, Frozen w_r: 0.443039, Best w_c: 2.408670


Add model predictions to dataframe

In [9]:
data_with_preds = data.copy()
for model_type in model.model_types:
    best_params, best_nll = params_and_nlls[model_type]
    predict_fn = model.get_vmap_predictor(model_type)
    preds = predict_fn(data_scenario_idx, data_action, data_closeness, *best_params)
    data_with_preds[f"{model_type}_pred"] = preds

# Save the updated dataframe
data_with_preds.to_csv("../data/planning-1/main_trials_tidy_with_preds.csv", index=False)
